# 01 · Del origen a RAW: conservar evidencia

**Objetivo:** explicar proyección, snapshot, lote, conteo y hash sin conectarse a SQL Server.

**Modalidad:** material para explicar en clase. Los ejemplos se ejecutan únicamente en memoria, con datos ficticios. No conectan a una base, no crean archivos ni ejecutan una ingesta. Los bloques SQL son ilustrativos y no se envían a ningún servidor.

**Prerrequisitos:** Python con pandas. Cada notebook es independiente: no requiere scripts `.py` ni haber ejecutado otra notebook.


In [ ]:
# Importar conexión centralizada desde el módulo
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO

# Obtener los engines
engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas correctamente.')
print(f'Servidor: {SERVIDOR}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')


## ¿Por qué una capa RAW?

RAW conserva lo recibido antes de normalizar o corregir. Cada registro necesita
origen, identificador de lote, fecha de extracción y evidencia de integridad.
Un hash detecta cambios; no demuestra que el contenido sea verdadero.

En un sistema real las tablas relacionadas deberían leerse con una estrategia
consistente. SERIALIZABLE puede bloquear escritores; SNAPSHOT requiere
configuración del DBA. No basta extraerlas a horas arbitrarias.


In [ ]:
consulta_ilustrativa = '''
SELECT InvoiceLineID, InvoiceID, StockItemID, Quantity,
       UnitPrice, TaxRate, TaxAmount, ExtendedPrice, LastEditedWhen
FROM Sales.InvoiceLines;
'''
print(consulta_ilustrativa)
# Es texto de ejemplo: no se ejecuta en una base.


In [ ]:
import hashlib
import json
import pandas as pd

origen = [
    {"InvoiceLineID": 1, "InvoiceID": 100, "StockItemID": 10,
     "Quantity": 2, "UnitPrice": "10.00", "TaxAmount": "4.20", "ExtendedPrice": "24.20"},
    {"InvoiceLineID": 2, "InvoiceID": 101, "StockItemID": 20,
     "Quantity": -1, "UnitPrice": "5.00", "TaxAmount": "-1.05", "ExtendedPrice": "-6.05"},
]
lote = "CLASE-001"
raw = []
for posicion, registro in enumerate(origen, start=1):
    payload = json.dumps(registro, sort_keys=True)
    raw.append({"lote": lote, "tabla_origen": "Sales.InvoiceLines",
                "fila": posicion, "payload": payload,
                "hash": hashlib.sha256(payload.encode("utf-8")).hexdigest()})
display(pd.DataFrame(raw))
print("Conteo extraído:", len(raw))


## Lectura del resultado

Esperamos dos registros, cada uno con el mismo lote y un hash propio. Los
importes llegan como texto para evitar conversiones accidentales a float.
El ejemplo incluye una nota de crédito: los signos negativos no son por sí
solos un error.

Este cuaderno no guarda RAW en disco. En una implementación real se conservaría
de forma persistente, con acceso restringido y retención acordada.

### Carga inicial versus incremental

Un snapshot completo relee el conjunto; una extracción incremental requiere
definir marcas, cambios y bajas. No confundir **no duplicar al cargar** con
**leer incrementalmente**. La marca solo avanzaría tras una publicación exitosa.


## Actividad de discusión

¿Se debe corregir un importe directamente en RAW? **No:** conservarlo permite reproducir el diagnóstico; la corrección debe tener evidencia separada.